In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_ROOT = Path("../data/sample/processed")

# ============================================================================
# Load Stage 4 Outputs
# ============================================================================

instance_labels = np.load(
    DATA_ROOT /
    "stage_4_instance_segmentation" /
    "instance_labels.npy"
)

# ============================================================================
# Load Stage 5 Outputs
# ============================================================================

cells_df = pd.read_csv(
    DATA_ROOT /
    "stage_5_cell_detection" /
    "cells.csv"
)

print("Instance Labels")
print("----------------")
print("Shape :", instance_labels.shape)
print("Dtype :", instance_labels.dtype)
print("Cells :", instance_labels.max())

print()

print(f"Detected cells : {len(cells_df)}")

display(cells_df.head())

Instance Labels
----------------
Shape : (64, 256, 256)
Dtype : int32
Cells : 199

Detected cells : 199


,cell_id,centroid_z,centroid_y,centroid_x,volume_voxels,z_min,y_min,x_min,z_max,y_max,x_max
0,1,0.231183,6.354839,52.204301,186.0,0,0,46,2,14,60
1,2,0.153153,6.815315,71.774775,222.0,0,0,65,2,16,80
2,3,0.320833,26.012500,65.191667,240.0,0,18,59,4,35,72
3,4,1.074627,62.743555,58.230665,737.0,0,55,48,4,72,68
4,5,0.364431,98.571429,34.836735,343.0,0,89,28,2,109,43


In [2]:
import zarr

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = (
        PROJECT_ROOT
        / "data" / "sample"
        / "biohub_5samples_20timepoints"
        / "train"
)

SAMPLE_ID = "44b6_0113de3b"

ZARR_PATH = (
        DATA_ROOT
        / SAMPLE_ID
        / f"{SAMPLE_ID}.zarr"
)

print(ZARR_PATH)
print(ZARR_PATH.exists())

ARRAY_PATH = ZARR_PATH / "0"

volume = zarr.open_array(
    str(ARRAY_PATH),
    mode="r"
)


D:\Projects\Kaggle\cell-tracking\data\sample\biohub_5samples_20timepoints\train\44b6_0113de3b\44b6_0113de3b.zarr
True


In [3]:
# ============================================================================
# Load Original Timepoint
# ============================================================================

TIMEPOINT = 0

original_volume = volume[TIMEPOINT]

print("Original Volume")
print("---------------")
print("Shape :", original_volume.shape)
print("Dtype :", original_volume.dtype)

Original Volume
---------------
Shape : (64, 256, 256)
Dtype : uint16


Extract Features

In [4]:
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull
from scipy.ndimage import center_of_mass
from skimage.measure import regionprops

# ============================================================================
# Feature Extraction
# ============================================================================

feature_rows = []

regions = {r.label: r for r in regionprops(instance_labels, intensity_image=original_volume)}

for cell_id in cells_df["cell_id"]:

    mask = instance_labels == cell_id
    coords = np.argwhere(mask)           # (z, y, x)
    intensities = original_volume[mask]

    region = regions[cell_id]

    # ------------------------------------------------------------------
    # Size
    # ------------------------------------------------------------------

    volume = mask.sum()

    zmin, ymin, xmin = coords.min(axis=0)
    zmax, ymax, xmax = coords.max(axis=0)

    bbox_depth = zmax - zmin + 1
    bbox_height = ymax - ymin + 1
    bbox_width = xmax - xmin + 1

    bbox_volume = bbox_depth * bbox_height * bbox_width

    extent = volume / bbox_volume

    # ------------------------------------------------------------------
    # PCA Shape
    # ------------------------------------------------------------------

    centered = coords - coords.mean(axis=0)

    if len(coords) >= 3:
        cov = np.cov(centered.T)
        eigvals, eigvecs = np.linalg.eigh(cov)
        eigvals = np.sort(eigvals)[::-1]
    else:
        eigvals = np.zeros(3)

    axis_major = np.sqrt(eigvals[0]) if eigvals[0] > 0 else 0
    axis_middle = np.sqrt(eigvals[1]) if eigvals[1] > 0 else 0
    axis_minor = np.sqrt(eigvals[2]) if eigvals[2] > 0 else 0

    elongation = (
        axis_major / axis_middle
        if axis_middle > 0 else 0
    )

    flatness = (
        axis_middle / axis_minor
        if axis_minor > 0 else 0
    )

    anisotropy = (
        axis_major / axis_minor
        if axis_minor > 0 else 0
    )

    # ------------------------------------------------------------------
    # Convex Hull
    # ------------------------------------------------------------------

    if len(coords) >= 4:
        try:
            hull = ConvexHull(coords)
            convex_volume = hull.volume
            solidity = volume / convex_volume if convex_volume > 0 else 1
        except Exception:
            convex_volume = np.nan
            solidity = np.nan
    else:
        convex_volume = np.nan
        solidity = np.nan

    # ------------------------------------------------------------------
    # Equivalent Sphere
    # ------------------------------------------------------------------

    equivalent_radius = (3 * volume / (4 * np.pi)) ** (1 / 3)

    # ------------------------------------------------------------------
    # Surface Approximation
    # ------------------------------------------------------------------

    surface_area = region.area_bbox

    compactness = (
        volume / (surface_area ** 1.5)
        if surface_area > 0 else 0
    )

    # ------------------------------------------------------------------
    # Centroid
    # ------------------------------------------------------------------

    cz, cy, cx = region.centroid

    # ------------------------------------------------------------------
    # Intensity
    # ------------------------------------------------------------------

    feature_rows.append({

        # ID
        "cell_id": cell_id,

        # Position
        "centroid_z": cz,
        "centroid_y": cy,
        "centroid_x": cx,

        # --------------------------
        # Intensity
        # --------------------------

        "intensity_mean": intensities.mean(),
        "intensity_median": np.median(intensities),
        "intensity_std": intensities.std(),
        "intensity_min": intensities.min(),
        "intensity_max": intensities.max(),
        "intensity_q25": np.percentile(intensities, 25),
        "intensity_q75": np.percentile(intensities, 75),

        # --------------------------
        # Size
        # --------------------------

        "volume": volume,

        "bbox_depth": bbox_depth,
        "bbox_height": bbox_height,
        "bbox_width": bbox_width,

        "extent": extent,

        "equivalent_radius": equivalent_radius,

        # --------------------------
        # Shape
        # --------------------------

        "axis_major": axis_major,
        "axis_middle": axis_middle,
        "axis_minor": axis_minor,

        "elongation": elongation,
        "flatness": flatness,
        "anisotropy": anisotropy,

        "convex_volume": convex_volume,
        "solidity": solidity,

        "compactness": compactness,

    })

features_df = pd.DataFrame(feature_rows)

overlap = features_df.columns.intersection(cells_df.columns).difference(["cell_id"])

cells_df = cells_df.drop(columns=overlap).merge(
    features_df,
    on="cell_id",
    how="left"
)

print(f"Extracted features for {len(cells_df)} cells")

display(cells_df.head())

Extracted features for 199 cells


,cell_id,volume_voxels,z_min,y_min,x_min,z_max,y_max,x_max,centroid_z,centroid_y,...,equivalent_radius,axis_major,axis_middle,axis_minor,elongation,flatness,anisotropy,convex_volume,solidity,compactness
0,1,186.0,0,0,46,2,14,60,0.231183,6.354839,...,3.541127,3.312989,2.928501,0.420773,1.131292,6.959811,7.873577,77.166667,2.410367,0.023965
1,2,222.0,0,0,65,2,16,80,0.153153,6.815315,...,3.756253,4.082647,3.323817,0.354509,1.228301,9.375832,11.516344,88.000000,2.522727,0.021110
2,3,240.0,0,18,59,4,35,72,0.320833,26.012500,...,3.855146,3.955908,2.965969,0.548120,1.333766,5.411165,7.217226,171.666667,1.398058,0.009131
3,4,737.0,0,55,48,4,72,68,1.074627,62.743555,...,5.603503,4.670576,3.679390,0.964733,1.269389,3.813895,4.841316,520.500000,1.415946,0.014695
4,5,343.0,0,89,28,2,109,43,0.364431,98.571429,...,4.342453,4.462224,3.304942,0.480760,1.350167,6.874411,9.281601,154.833333,2.215285,0.023338
